# 50 · API keys and data handling
Level L5 · Part 6 · core · ~25 min · builds on 07

## Goal
Find a leaked key in a copy of Scout's repo with a scanner you write, keep keys out of code, logs and
error messages, give each environment its own key, rotate a leaked key in the right order, and build
Scout's requests so the provider does not store them (`store=False`), knowing what each provider keeps
and where.

## The idea
In lesson 07 the key moved from code into `.env`. That is step one. A key is a password that also
spends money, so you also need to know three things. **Where it lives**: the environment or a
**secret manager**, never code, logs or error text. **How far it reaches**: one key per environment
(dev, staging, prod), so a leaked laptop key cannot touch production. **What to do when it leaks**:
**rotate** it (new key in, old key revoked). The same care applies to your users' data: every request
goes to someone else's servers, so you choose what they store, for how long, and in which country.

Analogy: a hotel key card. Each guest gets their own, it opens only their floor, and a lost card is
cancelled at the desk, not hunted for.

## Picture
```mermaid
flowchart LR
    SM["secret manager / .env<br/>(one key per environment)"] --> ENV["os.environ"]
    ENV --> S["Settings<br/>(key as SecretStr)"]
    S --> C["SDK client"]
    C -->|"request, store=False"| P["provider<br/>(workspace / project,<br/>spend cap, region)"]
    S -.->|"masked only"| LOG["logs"]
    REPO["repo"] --> SCAN["leak scanner"] -->|"found"| ROT["rotate:<br/>new key, deploy, revoke old"]
```

## Step by step
### 1. Fake keys only
This lesson never touches your real key. Every key below is a **fake** built at run time, in the same
shape as a real one so the scanner has something to find. We never print a key either: `mask` shows
the first four characters and the length, and `fingerprint` gives a short hash that is safe to log
and compare.

In [1]:
import hashlib, os, re, tempfile
from pathlib import Path

FAKE_GEMINI_KEY = "AIza" + "FAKE" + "x7Q2" * 7 + "abc"      # 39 characters, like a real Google key
FAKE_OPENAI_KEY = "sk-" + "proj-" + "FAKE" + "k9" * 12

def mask(secret: str) -> str:
    return f"{secret[:4]}... ({len(secret)} chars)"

def fingerprint(secret: str) -> str:
    return "fp:" + hashlib.sha256(secret.encode()).hexdigest()[:10]

print(mask(FAKE_GEMINI_KEY), fingerprint(FAKE_GEMINI_KEY))
print(mask(FAKE_OPENAI_KEY), fingerprint(FAKE_OPENAI_KEY))

AIza... (39 chars) fp:b4f74a7512
sk-p... (36 chars) fp:0e1227c268


A fingerprint lets you say "prod and dev use different keys" or "the key in this log line is the
leaked one" without ever showing the key.

### 2. A copy of Scout's repo with a leak in it
A teammate was in a hurry. We build their version of Scout in a **temporary folder**: the key pasted
into `scout.py`, an OpenAI key in a config file, and a `.env` that `.gitignore` forgot to list.

In [2]:
NOTES = ["Mars has two small moons, Phobos and Deimos.", "Venus spins backwards compared with most planets."]

repo = Path(tempfile.mkdtemp(prefix="scout_repo_"))
(repo / "scout.py").write_text(
    "from google import genai\n"
    f"client = genai.Client(vertexai=True, api_key={FAKE_GEMINI_KEY!r})  # TODO remove\n")
(repo / "config.yaml").write_text(f"model: gemini-3.5-flash-lite\nopenai_key: {FAKE_OPENAI_KEY}\n")
(repo / ".env").write_text(f"GOOGLE_CLOUD_API_KEY={FAKE_GEMINI_KEY}\n")
(repo / ".gitignore").write_text(".venv/\n__pycache__/\n")
(repo / "notes.txt").write_text("\n".join(NOTES) + "\n")
print(sorted(p.name for p in repo.iterdir()))

['.env', '.gitignore', 'config.yaml', 'notes.txt', 'scout.py']


### 3. Write a leak scanner
Real key formats have fixed prefixes, so a regular expression finds most of them. Google keys start
with `AIza`, OpenAI keys with `sk-`, Anthropic keys with `sk-ant-`. The scanner reads every file and
reports file, line, kind and the **masked** match. Like Git, it skips files `.gitignore` lists, since
they are never committed. It also checks that `.gitignore` lists `.env`.

In [3]:
KEY_PATTERNS = {
    "google": re.compile(r"AIza[0-9A-Za-z_-]{35}"),
    "openai": re.compile(r"sk-(?!ant-)[A-Za-z0-9_-]{20,}"),
    "anthropic": re.compile(r"sk-ant-[A-Za-z0-9_-]{20,}"),
}

def scan_repo(root: Path) -> list[dict]:
    findings = []
    ignored = (root / ".gitignore").read_text().split() if (root / ".gitignore").exists() else []
    for path in sorted(root.rglob("*")):
        if not path.is_file() or path.name in ignored:
            continue
        for lineno, line in enumerate(path.read_text(errors="ignore").splitlines(), 1):
            for kind, pattern in KEY_PATTERNS.items():
                for m in pattern.finditer(line):
                    findings.append({"file": path.name, "line": lineno, "kind": kind, "match": mask(m.group())})
    if (root / ".env").exists() and ".env" not in ignored:
        findings.append({"file": ".gitignore", "line": 0, "kind": "config", "match": ".env is not ignored"})
    return findings

Run it on the teammate's repo.

In [4]:
for f in scan_repo(repo):
    print(f"{f['file']}:{f['line']:<2} {f['kind']:<7} {f['match']}")

.env:1  google  AIza... (39 chars)
config.yaml:2  openai  sk-p... (36 chars)
scout.py:2  google  AIza... (39 chars)
.gitignore:0  config  .env is not ignored


Four findings. The key in `.env` is not a problem on its own: `.env` is the right place. The problem
is that `.gitignore` does not list it, so the next `git add .` commits it. Real teams run a scanner
like this (gitleaks, trufflehog, GitHub secret scanning) as a pre-commit hook and in CI.

### 4. Fix the repo
The key comes out of the code, the config keeps only non-secret settings, and `.env` is ignored.
The key is read from the environment at run time, as in lesson 07.

In [5]:
(repo / "scout.py").write_text(
    "import os\nfrom google import genai\n"
    'client = genai.Client(vertexai=True, api_key=os.environ["GOOGLE_CLOUD_API_KEY"])\n')
(repo / "config.yaml").write_text("model: gemini-3.5-flash-lite\n")
(repo / ".gitignore").write_text(".env\n.venv/\n__pycache__/\n")
print("findings after fix:", scan_repo(repo))

findings after fix: []


Clean. Git now ignores `.env`.

But if the key had already been **pushed**, deleting it from the file is not enough: it is still in
Git history and in every clone. A pushed key is a leaked key. Step 8 rotates it.

### 5. Where the key comes from, and who wins
`load_dotenv` (lesson 07) never overwrites a variable your shell already set. So in production, where
the platform injects the variable from a secret manager, a stray `.env` file cannot change the key.
We test this inside `patch.dict(os.environ, clear=True)`: an empty, throwaway environment, so the
real one (and any real key in it) is never read.

In [6]:
from unittest.mock import patch
from dotenv import load_dotenv

with patch.dict(os.environ, {}, clear=True):
    load_dotenv(repo / ".env")                                   # the fake .env in the temp repo
    print("from .env only    :", fingerprint(os.environ["GOOGLE_CLOUD_API_KEY"]))
with patch.dict(os.environ, {"GOOGLE_CLOUD_API_KEY": FAKE_GEMINI_KEY[::-1]}, clear=True):
    load_dotenv(repo / ".env")                                   # shell value already set
    print("shell value wins  :", fingerprint(os.environ["GOOGLE_CLOUD_API_KEY"]))

from .env only    : fp:b4f74a7512
shell value wins  : fp:dd9466101e


Two different fingerprints: the second run kept the shell's value. Outside the `with` block the
environment is back to what it was, and the real environment was never read.

### 6. A settings object that cannot leak the key
Printing a settings dict prints the key. Pydantic's `SecretStr` (pydantic from lesson 20) hides the
value in `print`, `repr` and `model_dump`; you get it only by calling `get_secret_value()`, at the one
place that needs it: the client.

In [7]:
from pydantic import BaseModel, SecretStr

class Settings(BaseModel):
    env: str
    model: str
    api_key: SecretStr

    @classmethod
    def from_env(cls, env: str) -> "Settings":
        key = os.environ.get("GOOGLE_CLOUD_API_KEY")
        if not key:
            raise RuntimeError("GOOGLE_CLOUD_API_KEY is not set (check .env or your secret manager)")
        return cls(env=env, model=os.environ.get("TUT_LLM_MODEL", "gemini-3.5-flash-lite"), api_key=key)

with patch.dict(os.environ, {"GOOGLE_CLOUD_API_KEY": FAKE_GEMINI_KEY}, clear=True):
    settings = Settings.from_env("dev")
print(settings)
print(settings.model_dump())
print("fingerprint for the log:", fingerprint(settings.api_key.get_secret_value()))

env='dev' model='gemini-3.5-flash-lite' api_key=SecretStr('**********')
{'env': 'dev', 'model': 'gemini-3.5-flash-lite', 'api_key': SecretStr('**********')}
fingerprint for the log: fp:b4f74a7512


The error message names the variable, never a value. Lesson 49's `redact` is the second net: it
replaces anything key-shaped that still reaches a log line.

### 7. One key per environment
Scout runs in three places. Each gets its own key, made in its own **Anthropic workspace** or **Google
Cloud project**, so each has its own spend cap, rate limits and usage report. A Gemini key belongs to
a project and every key in the project shares the project's monthly spend cap, so "separate cap"
means "separate project".

In [8]:
ENVIRONMENTS = {   # where each key is made, and its monthly cap (USD)
    "dev":     {"gemini_project": "scout-dev",     "anthropic_workspace": "Scout - Development", "cap": 20},
    "staging": {"gemini_project": "scout-staging", "anthropic_workspace": "Scout - Staging",     "cap": 50},
    "prod":    {"gemini_project": "scout-prod",    "anthropic_workspace": "Scout - Production",  "cap": 500},
}
keys = {env: "AIza" + "FAKE" + hashlib.sha256(env.encode()).hexdigest()[:31] for env in ENVIRONMENTS}
for env, where in ENVIRONMENTS.items():
    print(f"{env:<8} {where['gemini_project']:<14} cap ${where['cap']:<4} key {fingerprint(keys[env])}")
print("all keys different:", len({fingerprint(k) for k in keys.values()}) == len(keys))

dev      scout-dev      cap $20   key fp:d5715a3a64
staging  scout-staging  cap $50   key fp:23a1163bba
prod     scout-prod     cap $500  key fp:5a2f716a2d
all keys different: True


Now a dev key that leaks can burn at most $20 and cannot read production's stored files or batches.

### 8. Rotate a leaked key, in the right order
Rotation means: create a new key, deploy it, check it works, then revoke the old one. Revoke first
and production is down until the deploy lands. A toy provider shows the order. It plays the
provider's console; it is not a real API.

In [9]:
class ToyProvider:
    """Plays a provider console: keys are stored as fingerprints and can be revoked."""
    def __init__(self):
        self.active, self.counter = set(), 0
    def create_key(self, workspace: str) -> str:
        self.counter += 1
        key = "AIza" + "FAKE" + hashlib.sha256(f"{workspace}{self.counter}".encode()).hexdigest()[:31]
        self.active.add(fingerprint(key))
        return key
    def revoke(self, key: str):
        self.active.discard(fingerprint(key))
    def call(self, key: str) -> str:
        if fingerprint(key) not in self.active:
            raise PermissionError(f"401: API key {fingerprint(key)} is not valid")
        return "200 OK"

Production has one key. It leaks. We rotate.

In [10]:
provider = ToyProvider()
old = provider.create_key("Scout - Production")
deployed = old
new = provider.create_key("Scout - Production")      # 1. create the new key
deployed = new                                        # 2. deploy it (secret manager + restart)
print("3. new key works:", provider.call(deployed))   # 3. check
provider.revoke(old)                                  # 4. revoke the old one
try:
    provider.call(old)
except PermissionError as e:
    print("4. old key:", e)

3. new key works: 200 OK
4. old key: 401: API key fp:6db02a20cb is not valid


The leaked key is dead and production never stopped. Afterwards, read the provider's usage report for
the leak window: a spike there means someone used the key.

### 9. What the provider stores
Now the user's data. Some APIs keep each turn on the provider's servers so you can chain from it
(lesson 09). Scout keeps its own history, so it does not need that. The table below is what each
provider says today (see RESEARCH.md section 34).

In [11]:
DATA_POLICY = [
    ("OpenAI Responses",          "stores turns by default, 30 days", "store=False"),
    ("Gemini Interactions",       "stores by default, 55 days paid / 1 day free", "store=False"),
    ("Gemini generate_content",   "no chaining store; abuse logs for a limited time", "-"),
    ("Anthropic Messages",        "no chaining store; logs kept for a limited time (see its retention policy)", "-"),
    ("Vertex AI (any Gemini)",    "24 h in-memory cache per project", "disable cache per project"),
]
width = max(len(default) for _, default, _ in DATA_POLICY)
for api, default, opt_out in DATA_POLICY:
    print(f"{api:<25} | {default:<{width}} | {opt_out}")

OpenAI Responses          | stores turns by default, 30 days                                           | store=False
Gemini Interactions       | stores by default, 55 days paid / 1 day free                               | store=False
Gemini generate_content   | no chaining store; abuse logs for a limited time                           | -
Anthropic Messages        | no chaining store; logs kept for a limited time (see its retention policy) | -
Vertex AI (any Gemini)    | 24 h in-memory cache per project                                           | disable cache per project


### 10. Build Scout's requests with `store=False`, and check them
We build the request bodies by hand (no call is sent) and add a guard that refuses to send a
request that would be stored. The guard runs before every call in production Scout.

In [12]:
question = "How many moons does Mars have?"
openai_request = {"model": "gpt-6-luna", "input": question, "store": False}
gemini_interaction = {"model": "gemini-3.5-flash-lite", "input": question, "store": False}

def check_not_stored(api: str, body: dict) -> dict:
    if api in {"openai.responses", "gemini.interactions"} and body.get("store", True) is not False:
        raise ValueError(f"{api}: request would be stored on the provider (set store=False)")
    return body

for api, body in [("openai.responses", openai_request), ("gemini.interactions", gemini_interaction)]:
    print(api, "ok:", sorted(check_not_stored(api, body)))

openai.responses ok: ['input', 'model', 'store']
gemini.interactions ok: ['input', 'model', 'store']


`store` is a request field in both APIs; the default is `True`, so leaving it out means "store it".
The guard treats a missing field as stored.

### 11. Which Gemini, and where
The same model has different data terms depending on how you reach it. This decides which one Scout
may use with real user data.

In [13]:
GEMINI_ROUTES = {
    "AI Studio, free tier":   {"used to improve Google products": True,  "region you choose": False},
    "Gemini API, paid":       {"used to improve Google products": False, "region you choose": False},
    "Vertex AI (Google Cloud)": {"used to improve Google products": False, "region you choose": True},
}
for route, terms in GEMINI_ROUTES.items():
    trains, region = terms["used to improve Google products"], terms["region you choose"]
    verdict = "demo data only" if trains else (
        "user data, region you pick (not the global endpoint)" if region else "user data, any country")
    print(f"{route:<25} training: {trains!s:<5} region: {region!s:<5} -> {verdict}")

AI Studio, free tier      training: True  region: False -> demo data only
Gemini API, paid          training: False region: False -> user data, any country
Vertex AI (Google Cloud)  training: False region: True  -> user data, region you pick (not the global endpoint)


On the free tier, human reviewers may read prompts, and Google says not to send personal data. The
paid Gemini API may cache data "in any country". Vertex AI lets you pick the location. A regional or
multi-region endpoint keeps data at rest and ML processing in that region or jurisdiction; the `global`
endpoint gives no residency guarantee. That is what data-residency rules (for example EU-only) need.
That is why this course uses Vertex.

Careful: Scout's `llm_setup.py` passes no `location`, so the client uses the `global` endpoint unless
the `GOOGLE_CLOUD_LOCATION` variable is set (for example in your `.env`). The course default is **not**
region-pinned. For real user data, set it to a region such as `europe-west4` or a multi-region such as
`eu`, and check it isn't `global`. (The `eu` multi-region excludes the UK and Switzerland.)

### 12. Clean up
Delete the temporary repo. Nothing was written inside the topic folder.

In [14]:
import shutil
shutil.rmtree(repo)
print("temp repo removed:", not repo.exists())

temp repo removed: True


## At work
- **Secret managers, not files, in production.** Keys live in Google Secret Manager, AWS Secrets
  Manager or Vault, arrive as environment variables, one key per service. (beyond this course)
- **Spend caps are the alarm for a leaked key.** Per-workspace (Anthropic) and per-project (Gemini)
  monthly caps limit a stolen key's bill; Gemini's cap can be overshot as billing lags. → lesson 54
- **Keys leak through tools, not only Git.** An agent that can read files or run shell commands can be
  tricked into reading `.env` and sending it out. → lesson 51
- **Zero data retention is a checklist, not a flag.** On Vertex: no request logging, caching off,
  `store=false`, an abuse-monitoring exception, no Google Search grounding (its logs are kept up to 3
  days). (beyond this course)

## What just happened
- A regex scanner found three fake keys (two hard-coded, one in an unignored `.env`) and one
  `.gitignore` gap in a temp copy of Scout's repo, and printed only masked matches.
- Reading the key from the environment fixed the code; `load_dotenv` showed the shell's value wins over `.env`.
- `SecretStr` kept the key out of `print` and `model_dump`; fingerprints let us compare keys without showing them.
- Each environment got its own key in its own workspace or project, with its own spend cap.
- Rotation went create, deploy, check, revoke: the old key failed and production never stopped.
- Scout's requests carry `store=False`, a guard rejects stored requests, and Vertex AI, on a regional
  endpoint, is the route for user data.

## Common mistakes
**1. Putting the key in an error message.** It feels helpful while debugging, but error text lands
in logs, trace tools and bug reports. Here a guard catches it before the line is logged (the guard's
own message is masked).

In [15]:
def assert_no_secret(line: str) -> str:
    for kind, pattern in KEY_PATTERNS.items():
        if m := pattern.search(line):
            raise ValueError(f"log line contains a {kind} key ({mask(m.group())})")
    return line

key = settings.api_key.get_secret_value()
assert_no_secret(f"auth failed for key {key}, retrying")

ValueError: log line contains a google key (AIza... (39 chars))

Fix: log the fingerprint, never the value.

In [16]:
print(assert_no_secret(f"auth failed for key {fingerprint(key)}, retrying"))

auth failed for key fp:b4f74a7512, retrying


**2. Passing a `SecretStr` where the SDK wants the string.** `str(secret)` is the stars, so the
provider receives `**********` and rejects it.

In [17]:
provider = ToyProvider()
settings = Settings(env="prod", model="gemini-3.5-flash-lite", api_key=provider.create_key("Scout - Production"))
print(provider.call(str(settings.api_key)))

PermissionError: 401: API key fp:6e9ba2d474 is not valid

Fix: call `get_secret_value()` once, where the client is made.

In [18]:
print(provider.call(settings.api_key.get_secret_value()))

200 OK


**3. Forgetting `store` and assuming the default is off.** For OpenAI Responses and Gemini
Interactions the default is to store.

In [19]:
check_not_stored("openai.responses", {"model": "gpt-6-luna", "input": question})

ValueError: openai.responses: request would be stored on the provider (set store=False)

Fix: set `"store": False` on every request (as in step 10), and keep the guard.

## Production checklist
- [ ] No key in code, config or notebooks; a secret scanner runs as a pre-commit hook and in CI.
- [ ] `.env` is in `.gitignore`; production reads keys from a secret manager.
- [ ] One key per environment, each in its own workspace or project, each with a spend cap.
- [ ] Logs and errors show fingerprints or masks only; lesson 49's `redact` runs on every log line.
- [ ] A written rotation runbook: create, deploy, check, revoke, then read usage for the leak window.
- [ ] Every request sets `store=False` where the API stores by default; a guard enforces it.
- [ ] User data goes only to a paid, no-training route in an approved, non-global location (here Vertex AI).

## Try it
Add a scanner rule for GitHub personal access tokens (`ghp_` followed by 36 letters and digits).
Build a fake one at run time, write a temp repo with it in `deploy.sh`, and show that `scan_repo`
reports it with a masked match. Clean up the temp folder.

In [20]:
# your code here

## Key terms
- **secret manager** — a service that stores keys encrypted, controls who reads them, and injects them into apps.
- **secret scanning** — searching files and Git history for key-shaped strings, in a pre-commit hook and in CI.
- **key fingerprint** — a short hash of a key: safe to log and compare, cannot be turned back into the key.
- **key rotation** — replacing a key: create the new one, deploy, check, then revoke the old one.
- **per-environment keys** — a separate key, workspace or project and spend cap for dev, staging and prod.
- **workspace (Anthropic)** — a group in an Anthropic organisation with its own keys, spend limit and rate limits.
- **data residency** — data is stored and processed only in a chosen region; on Vertex AI, a `location` that is not `global`.
- **zero data retention** — the provider keeps no prompts or replies; on Vertex it takes several settings, not one flag.